# Supplementary Table S2 somatic SNV/indel summary

This notebook compiles the somatic SNV/indel summary tables used for Supplementary Table S2 from filtered and annotated mutation calls generated from the upstream Mutect2 -> vcf2maf -> OncoKB workflow.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import ast
import re
import seaborn as sns
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import matplotlib as mpl  
from collections import OrderedDict

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 120) 

In [ ]:
new_sample_list = \
['RAP-001_65852_T',
 'RAP-001_65847_T',
 'RAP-002_14015_T',
 'RAP-002_14071_T',
 'RAP-002_14009_T',
 'RAP-002_14057_T',
 'RAP-002_14029_T',
 'RAP-003_69675_T',
 'RAP-003_69443_T',
 'RAP-003_69692_T',
 'RAP-004_70735_T',
 'RAP-004_70676_T',
 'RAP-004_70385_T',
 'RAP-004_70939_T',
 'RAP-004_70396_T',
 'RAP-004_70647_T',
 'RAP-004_70370_T',
 'RAP-004_70401_T',
 'RAP-005_179178_T',
 'RAP-005_179317_T',
 'RAP-005_179141_T',
 'RAP-005_179174_T',
 'RAP-005_179337_T',
 'RAP-005_179131_T',
 'RAP-005_179341_T',
 'RAP-006_150206_T',
 'RAP-006_150169_T',
 'RAP-006_150153_T',
 'RAP-006_150124_T',
 'RAP-006_150264_T',
 'RAP-006_150175_T',
 'RAP-006_150200_T',
 'RAP-006_150289_T',
 'RAP-006_150274_T',
 'RAP-006_150230_T',
 'RAP-006_150113_T',
 'RAP-006_150139_T',
 'RAP-006_150145_T',
 'RAP-007_245081_T',
 'RAP-007_245138_T',
 'RAP-007_244988_T',
 'RAP-007_244103_T',
 'RAP-007_245184_T',
 'RAP-007_244126_T',
 'RAP-007_245062_T',
 'RAP-007_244120_T',
 'RAP-007_245163_T',
 'RAP-007_245725_T']

In [ ]:
oncokb_dir = 'PATH_TO_ONCOKB_TABLES'

os.chdir(oncokb_dir)

temp_files = sorted([i for i in os.listdir(oncokb_dir) if i[:-4] in new_sample_list])

In [7]:
oncokb_annotated_dfs = {}

for file in temp_files:
    
    oncokb_annotated_dfs[file[:-4]] = pd.read_csv(oncokb_dir + file, sep='\t', low_memory=False)

In [12]:
all_var_types = \
["3'Flank",
 "3'UTR",
 "5'Flank",
 "5'UTR",
 'Frame_Shift_Del',
 'Frame_Shift_Ins',
 'IGR',
 'In_Frame_Del',
 'In_Frame_Ins',
 'Intron',
 'Missense_Mutation',
 'Nonsense_Mutation',
 'RNA',
 'Silent',
 'Splice_Region',
 'Splice_Site',
 'Targeted_Region',
 'Translation_Start_Site']

selected_var_types = \
['Frame_Shift_Del',
 'Frame_Shift_Ins',
 'In_Frame_Del',
 'In_Frame_Ins',
 'Missense_Mutation',
 'Nonsense_Mutation',
 'Splice_Site',
 'Translation_Start_Site']

In [13]:
to_save_dfs = {}

for sample_id in oncokb_annotated_dfs:

    temp_df = oncokb_annotated_dfs[sample_id]

    to_save_dfs[sample_id] = temp_df[temp_df['Variant_Classification'].isin(selected_var_types)]

In [ ]:
# --- mutation group definitions ---
point_mut_list = ["5'Flank", "Missense_Mutation"]
lof_mut_list = ["Nonsense_Mutation", "Frame_Shift_Del", "Frame_Shift_Ins", "Splice_Site"]
if_mut_list = ["In_Frame_Del", "In_Frame_Ins"]

mutation_group_map = {}
mutation_group_map.update({x: "Point_Mutation" for x in point_mut_list})
mutation_group_map.update({x: "Loss_of_Function_Mutation" for x in lof_mut_list})
mutation_group_map.update({x: "In_Frame_Mutation" for x in if_mut_list})

# --- columns to keep if present ---
preferred_cols = [
    "Patient_ID",
    "Sample_ID",
    "Mutation_Group",
    "Hugo_Symbol",
    "Chromosome",
    "Start_Position",
    "End_Position",
    "Reference_Allele",
    "Tumor_Seq_Allele2",
    "Variant_Classification",
    "Variant_Type",
    "HGVSc",
    "HGVSp_Short",
    "Transcript_ID",
    "t_depth",
    "t_alt_count",
    "VAF",
    "n_depth",
    "n_alt_count",
    "dbSNP_RS",
    "GENE_IN_ONCOKB",
    "VARIANT_IN_ONCOKB",
    "MUTATION_EFFECT",
]

to_save_dfs = {}

for key, temp_df in oncokb_annotated_dfs.items():
    df = temp_df.copy()

    # filter to selected variant classes
    df = df[df["Variant_Classification"].isin(selected_var_types)].copy()

    # parse patient/sample IDs from dict key
    # example: RAP-001_65847_T
    patient_id = key.split("_")[0]
    sample_id = key.rsplit("_", 1)[0]

    # add new columns
    df["Patient_ID"] = patient_id
    df["Sample_ID"] = sample_id
    df["Mutation_Group"] = df["Variant_Classification"].map(mutation_group_map).fillna("Other")

    # optional rename for cleaner export
    if "Tumor_Seq_Allele2" in df.columns:
        df = df.rename(columns={"Tumor_Seq_Allele2": "Alt_Allele"})
        preferred_cols_use = [
            c if c != "Tumor_Seq_Allele2" else "Alt_Allele"
            for c in preferred_cols
        ]
    else:
        preferred_cols_use = preferred_cols.copy()

    # keep only columns that actually exist
    keep_cols = [c for c in preferred_cols_use if c in df.columns]
    df = df[keep_cols].copy()

    to_save_dfs[sample_id] = df

# combine all samples into one dataframe for Table S2
table_s2_df = pd.concat(to_save_dfs.values(), ignore_index=True)

# sorting
mutation_group_order = {
    "Point_Mutation": 0,
    "Loss_of_Function_Mutation": 1,
    "In_Frame_Mutation": 2,
    "Other": 3,
}

table_s2_df["Mutation_Group_Order"] = (
    table_s2_df["Mutation_Group"].map(mutation_group_order).fillna(99)
)

table_s2_df = table_s2_df.sort_values(
    by=["Patient_ID", "Sample_ID", "Mutation_Group_Order", "Hugo_Symbol", "Chromosome", "Start_Position"],
    kind="stable"
).drop(columns="Mutation_Group_Order")

In [ ]:
table_s2_df.to_csv("PATH_TO_OUTPUT_DIR", sep="\t", index=False)